# Pitch Model with Quasi-Steady Aerodynamics

---

Hello! Welcome back to **Introduction to Aeroelastic Instabilities with Jupyter**.

In the previous notebook, you met your **first flutter**: by coupling heave and pitch into a 2-DOF typical section, we watched the eigenvalues cross into the unstable half-plane at a certain **flutter speed**. To get there, however, we made a deliberate simplification: we accounted for the aerodynamic effect of the *heave* velocity but **completely neglected the effect of the pitch rate** $\dot{\theta}$ on the aerodynamic forces, promising to come back to it. This is the notebook where we keep that promise.

We do it in the simplest possible setting: a **pitch-only** typical section, free to rotate about its elastic axis and nothing else. In other words, the dynamic sibling of the torsional-divergence model of Notebook 1. Working in a single degree of freedom lets us isolate the pitch-rate aerodynamics cleanly and answer two key questions:

1. When the airfoil rotates, the relative wind direction is **different at every point** along the chord. Which point should we use to define the effective angle of attack?
2. What happens to the **stability** of the system? Is it, like heave, always stable, or can pitch alone go unstable?

The answers contain a surprise. We will find that quasi-steady aerodynamics gives a stability verdict for the pitch-only section that **does not match the true physical behaviour**. That mismatch is exactly the motivation we need to introduce the unsteady aerodynamic theory of Theodorsen in **Notebook 5**.

### Learning Objectives

By the end of this notebook, you will be able to:

1. Explain how pitch-rate rotation creates a spatially varying effective angle of attack along the chord
2. Apply the quarter-chord vortex / three-quarter-chord collocation result from classical thin-airfoil theory to justify evaluating the quasi-steady angle of attack at the three-quarter chord
3. Derive the equation of motion for the pitch-only typical section using moment equilibrium about the elastic axis
4. Identify the aerodynamic damping and stiffness terms in the equation of motion, and explain their physical significance and sign
5. Implement a state-space simulation to compute the time response and eigenvalues of the pitch model across a range of flight speeds
6. Interpret the root locus and V-f/V-g plots to explain the stability behaviour of the pitch model for different elastic axis positions

*Let's get started!*

In [ ]:
# We start by importing necessary libraries
import numpy as np
import matplotlib.pyplot as plt

# Automatically use "inline" for Google Colab or "widget" for interactive plots in Jupyter
import sys
if 'google.colab' in sys.modules:
    %matplotlib inline
else:
    %matplotlib widget

## 1. The Physics: Pitch-Rate Contribution to the Effective Angle of Attack

In this notebook we return to the simplified **pitch** model that we used in Notebook 1 to introduce the aeroelastic polar and torsional divergence. In this model, the typical section is only allowed to rotate about its elastic axis, and we ignore any vertical translation. However, similarly to the heave-only model of Notebook 2, we treat it as a *dynamic* system. This means that the pitch angle $\theta(t)$, pitch rate $\dot{\theta}(t)$, and pitch acceleration $\ddot{\theta}(t)$ all vary in time. Also in this case we adopt **quasi-steady** aerodynamics, meaning that the aerodynamic forces depend only on the instantaneous angle of attack. The key new question is: how does pitch motion define an effective angle of attack?

---

### 1.1 A Key Difference from Heave

In the heave model, the airfoil translated straight up or down. At any given instant, **every point along the chord had the same vertical velocity** $\dot{h}$. This made the effective angle of attack simple: it was the same everywhere, equal to $-\dot{h}/V$.

Pitch motion is fundamentally different. When the airfoil **rotates** at an angular rate $\dot{\theta}$, each point along the chord acquires a different velocity perpendicular to the chord. A point close to the elastic axis barely moves, while a point far from it moves a lot.

Take a look at the figure below. During a nose-up rotation ($\dot{\theta} > 0$), points **forward** of the elastic axis (toward the leading edge) acquire an **upward** velocity, while points **aft** of the elastic axis (toward the trailing edge) acquire a **downward** velocity. As we know from the heave model, the airfoil sees a relative velocity opposite to its own motion, which creates an effective angle of attack $\alpha_{\dot{\theta}}$. But since the velocity is different at every point, **the effective angle of attack is also different at every point.**

<figure style="width:75%; margin: auto;">
<img src="../figures/04_airfoil_pitch_rate.svg" style="width:100%">
<figcaption style="text-align: center;">

Airfoil in nose-up pitch rotation: the perpendicular velocity $v_t = \dot{\theta}\,r$ is proportional to the distance $r$ from the elastic axis. Points forward of the EA move upward; points aft move downward.

</figcaption>
</figure>

### 1.2 The Perpendicular Velocity at Each Chord Point

Let us compute this velocity precisely. We use the two reference frames shown in the figure:

- The **inertial frame** $\left(\hat{\boldsymbol{i}}_1, \hat{\boldsymbol{i}}_2, \hat{\boldsymbol{i}}_3\right)$ fixed to the ground.

- The **body frame** $\left(\hat{\boldsymbol{b}}_1, \hat{\boldsymbol{b}}_2, \hat{\boldsymbol{b}}_3\right)$ fixed to the airfoil and moving with it.

When the airfoil rotates at rate $\dot{\theta}$, a chord point at position $r$ from the elastic axis in the direction of $\hat{\boldsymbol{b}}_1$ acquires a velocity **perpendicular to the chord** equal to:

$$\boldsymbol{v}_t = \dot{\theta} \hat{\boldsymbol{b}}_2 \times r \hat{\boldsymbol{b}}_1 = - \dot{\theta} r \hat{\boldsymbol{b}}_3.$$

The unit vector $\hat{\boldsymbol{b}}_3$ has components in the inertial frame given by:

$$\hat{\boldsymbol{b}}_3 = \cos\theta \hat{\boldsymbol{i}}_3 + \sin\theta \hat{\boldsymbol{i}}_1.$$

Consequently, the velocity of the chord point $r$ in the inertial frame is:

$$\boldsymbol{v}_t = - \dot{\theta} r \hat{\boldsymbol{b}}_3 = - \dot{\theta} r \left( \cos\theta \hat{\boldsymbol{i}}_3 + \sin\theta \hat{\boldsymbol{i}}_1 \right).$$

To simplify the expression, we use the small angles assumption, which allows us to approximate $\cos\theta \approx 1$ and $\sin\theta \approx 0$. This gives us:

$$\boldsymbol{v}_t \approx - \dot{\theta} r \hat{\boldsymbol{i}}_3.$$

This simplification allows us to calculate the effective angle of attack $\alpha_{\dot{\theta}}$ at each chord point as:

$$\alpha_{\dot{\theta}}(r) = \tan^{-1}\left(-\frac{v_{t}}{V}\right) \approx -\frac{v_{t}}{V} = \frac{\dot{\theta} r}{V}.$$

The effective angle of attack varies **linearly** along the chord, with a zero crossing exactly at the elastic axis. This is a direct consequence of rigid-body kinematics.

Now we face a problem: the lift model of thin-airfoil theory uses a single angle of attack. Instead, here the angle of attack is a different number at every chord point. Which one do we use?

### 1.3 The Three-Quarter Chord: Why That Specific Point?

The short answer to this question is: **the three-quarter chord**. This is not an arbitrary choice. It is the unique point that makes a simple single-vortex aerodynamic model exactly reproduce the correct lift. Let us see why.

In Notebook 1, we introduced the idea that thin airfoil theory obtains the total lift generated by an airfoil by modelling it as a continuous sheet of infinitesimal vortex elements distributed along the chord and by finding the distribution of vorticity $\gamma(x)$ that satisfies the flow boundary conditions. We also discussed the idea that a lumped vortex element placed at the centroid of the continuous vorticity distribution, that is to say at the quarter chord, produces the same total lift and the same pitching moment about any axis as the full distributed sheet.

The strength of this lumped vortex element corresponds to the so-called circulation $\Gamma$, which is the integral of the continuous vorticity distribution. Thin-airfoil theory shows that for a flat plate at angle of attack $\alpha$, this circulation is:

$$\Gamma = \pi c V \alpha.$$

If we want to determine $\Gamma$ starting from the lumped vortex model instead of integrating the distributed vorticity, we need to impose one physical condition. The natural choice is **flow tangency**: at the surface of the airfoil, the flow velocity perpendicular to the chord must be zero. In other words, no flow can pass through a solid surface. We need to enforce this condition at exactly one representative chord station.

A single point vortex of strength $\Gamma$ at the quarter chord induces a downwash velocity $w$ at a downstream distance $d$ given by the Biot–Savart law for a vortex filament:

$$w = -\frac{\Gamma}{2\pi d},$$

where the negative sign indicates that a clockwise (positive-lift) vortex induces a downward velocity aft of it. The flow tangency condition requires this downwash to cancel the normal component of the freestream:

$$w = -V\sin\alpha \approx -V\alpha,$$

where we have used the small angles approximation. Equating the two expressions for $w$ gives us:

$$\Gamma = 2\pi d\, V \alpha.$$

Comparing with the exact result $\Gamma = \pi c V \alpha$, we find out that we need to impose the flow tangency condition at $d = c/2$. Starting from the quarter chord, a distance of $c/2$ downstream lands exactly at the **three-quarter chord**.

> **The quarter-chord / three-quarter-chord duality:** placing the lumped vortex at the quarter chord and enforcing flow tangency at the three-quarter chord is the *unique* combination that exactly reproduces the correct thin-airfoil lift. The two points are inseparable partners.

### 1.4 Putting It Together: The Effective Angle of Attack of the Pitch Model

We now have all the ingredients. For quasi-steady aerodynamics, we evaluate the angle of attack at the **three-quarter chord** point T as shown in the figure below. Note that, from this point on we use the **half-chord** $b = c/2$ instead of the full chord $c$ as the reference length, which is the standard convention in aeroelasticity.

<figure style="width:75%; margin: auto;">
<img src="../figures/04_typical_section_pitch_dof.svg" style="width:100%">
<figcaption style="text-align: center;">

Typical section with pitch degree of freedom only. The elastic axis (EA) is at $eb$ from the midchord (positive toward the trailing edge). The aerodynamic center (AC) is at the quarter chord ($-b/2$ from midchord). The three-quarter chord collocation point (T) is at $+b/2$ from midchord.

</figcaption>
</figure>

From Section 1.2, the perpendicular velocity at T for a positive pitch rate $\dot{\theta}$ is:

$$v_t = - \dot{\theta} r = - \dot{\theta} \left(\frac{b}{2} - eb\right) = \dot{\theta} b\left(e - \frac{1}{2}\right),$$

where, according to our convention, the distance $r$ is positive aft of the elastic axis, in such a way that a positive pitch rate $\dot{\theta}$ produces a negative velocity $v_t$ at T.

Consequently, the effective angle of attack due to pitch rate is:

$$\alpha_{\dot{\theta}} \approx -\frac{v_t}{V} = \frac{\dot{\theta} b\left(\frac{1}{2} - e\right)}{V}.$$

Adding the quasi-static contribution from the pitch angle $\theta$, the **total effective angle of attack** is:

$$\boxed{\alpha = \theta + \frac{b}{V}\left(\dfrac{1}{2} - e\right)\dot{\theta}.}$$

## 2. Equation of Motion

---

We build the dynamic model by applying Newton's second law for rotation (moment equilibrium about the elastic axis). Three moments act on the typical section:

1. **Elastic restoring moment:** $-k_\theta \theta$ (opposes the rotation with rotational stiffness $k_\theta$).
2. **Aerodynamic moment:** Quasi-steady lift $L$ acts at the quarter chord and its moment arm from the AC to the EA is $eb - (-b/2) = b(1/2 + e)$.
3. **Inertial moment:** $I_{\rm EA} \ddot{\theta}$ (resists angular acceleration with mass moment of inertia $I_{\rm EA}$ about the elastic axis).

Newton's second law for rotation gives:

$$M_{\rm a} - k_\theta \theta = I_{\rm EA} \ddot{\theta}$$

where $M_{\rm a}$ is the aerodynamic moment about the elastic axis, which we can compute by multiplying the lift $L$ by its moment arm:

$$M_{\rm a} = L b\left(\frac{1}{2} + e\right).$$

Substituting $L = q c c_{l\alpha} \alpha$ and $\alpha = \theta + \frac{\dot{\theta} b\left(\frac{1}{2} - e\right)}{V}$, and rearranging (using $c = 2b$), we get:

$$M_{\rm a} = 2 q b c_{l\alpha} \left(\theta + \frac{\dot{\theta} b\left(\frac{1}{2} - e\right)}{V}\right) b\left(\frac{1}{2} + e\right) = q c_{l\alpha} b^2 \left(1 + 2e\right) \theta + \frac{q c_{l\alpha} b^3 \left(\frac{1}{2} - 2e^2\right)}{V} \dot{\theta}.$$

Rewriting so that all terms appear on the left-hand side gives us the equation of motion:

$$I_{\rm EA} \ddot{\theta} + \left(-\frac{q c_{l\alpha} b^3\left(\frac{1}{2} - 2e^2\right)}{V}\right) \dot{\theta} + \left(k_\theta - q c_{l\alpha} b^2\left(1 + 2e\right)\right) \theta = 0.$$

### STOP and Look! 👀

Similarly to the heave model, we can identify a second-order linear ordinary differential equation, with inertia, damping, and stiffness terms. However, as you can see, these terms look more complex now. Let's compare them to those of the heave model to understand their physical meaning and implications for stability.

| | Heave model | Pitch model |
|---|---|---|
| Inertia term | $m\ddot{h}$ | $I_{\rm EA}\ddot{\theta}$ |
| Damping term | $\dfrac{q c c_{l\alpha}}{V}$ | $-\frac{q c_{l\alpha} b^3\left(\frac{1}{2} - 2e^2\right)}{V}$ |
| Stiffness term | $k_h$ | $k_\theta - q c_{l\alpha} b^2\left(1 + 2e\right)$ |

In the heave model, the aerodynamic damping was always positive, meaning that the aerodynamics removed energy from the system, making it always stable, no matter the flight speed. **In the pitch model, the sign of the aerodynamic damping depends on the position of the elastic axis $e$**, and it becomes negative if $\left(1/2 - 2e^2\right) > 0$. This means that the system is **always unstable** when $\left|e\right| < 1/2$, that is to say when the elastic axis is located between the quarter and three-quarter chord. Conversely, when the elastic axis is located ahead of the quarter chord or behind the three-quarter chord, the system is intrinsically stable. Why does this happen in our pitch model?

> If $\left|e\right| < 1/2$ (EA between the quarter and three-quarter chords), two effects act simultaneously when the airfoil pitches **nose-up** $\left(\dot{\theta} > 0\right)$: (1) the lift acts *forward* of the elastic axis, producing a nose-up moment; (2) the three-quarter chord moves *downward*, increasing the effective angle of attack and thus the lift. Both effects reinforce each other in the **same direction** as the rotation rate $\dot{\theta}$. In other words, this is positive feedback: the aerodynamics induces more motion instead of opposing it and the system is dynamically unstable.

> If $e<-1/2$ (EA forward of the quarter chord), the lift acts *behind* the elastic axis, so any pitch-rate increase in lift produces a restoring (nose-down) moment. The aerodynamics actively **opposes** the rotation, and the system is always dynamically stable, with no divergence either, as we saw in Notebook 1.

> If $e>1/2$ (EA aft of the three-quarter chord), the lift force acts *ahead* of the elastic axis, but the three-quarter chord moves *upward*, which produces a decrease in the effective angle of attack and consequently in the lift. The pitch rate thus reduces the aerodynamic moment rather than amplifying it, resulting in a **net stabilising effect**, making the system dynamically stable.

The instability for $\left|e\right| < 1/2$ **is not flutter**, keep this distinction in mind. Flutter is a **speed-triggered** loss of stability: a system that is stable at low speed and turns unstable only once a critical airspeed is exceeded, where the damping *crosses* from positive to negative. That is exactly what you witnessed for the 2-DOF section in Notebook 3, and we will come back to the difference in the conclusion. Instead, in this case the *sign* of the damping is fixed by the $e$ alone, and it does not depend on the flight speed.

Analogously to the heave model, we can also observe that the magnitude of the damping term is proportional to the density $\rho$ and the flight speed $V$, which means that the system becomes either more stable or unstable as we fly faster and at lower altitudes.

As for the stiffness term, we can see that it is the sum of the elastic stiffness $k_\theta$ and of the aerodynamic stiffness $- q c_{l\alpha} b^2\left(1 + 2e\right)$. This sum should look familiar to you if you remember the torsional divergence model from Notebook 1. In fact, the pitch model is a dynamic extension of the torsional divergence model, where we have added inertia and damping terms to the equation of motion. If we set the sum of the stiffness terms equal to zero, we obtain a similar expression of the divergence dynamic pressure as in the torsional divergence model:

$$k_\theta - q c_{l\alpha} b^2\left(1 + 2e\right) = 0 \quad \Longrightarrow \quad q_D = \frac{k_\theta}{b^2\left(1 + 2e\right) c_{l\alpha}},$$

where this time we have used the half-chord $b$ as reference length and a different definition of the location of the elastic axis compared to Notebook 1. This expression tells us that the pitch model can experience static instability in addition to dynamic instability. Analogously to what we saw in Notebook 1, static instability tends to occur earlier (lower flight speed) as the elastic axis moves towards the trailing edge (larger moment arm of the lift force), and it can be completely avoided if the elastic axis is located ahead of the quarter chord $\left(e < -1/2\right)$.

Let's now translate the equation of motion into a state-space model and compute the time response and eigenvalues across a range of flight speeds.

## 3. State-Space Representation

---

We follow the same procedure as Notebook 2. We isolate the highest derivative:

$$I_{\rm EA} \ddot{\theta} = \frac{q c_{l\alpha} b^3\left(\frac{1}{2} - 2e^2\right)}{V}\,\dot{\theta} + \left(q c_{l\alpha} b^2\left(1 + 2e\right) - k_\theta\right)\theta,$$

and we define the state vector $\boldsymbol{x} = [\theta, \dot{\theta}]^\intercal$, so that we can write:

$$\dot{\boldsymbol{x}} = \boldsymbol{A}\boldsymbol{x}$$

$$\boldsymbol{A} = \begin{bmatrix} 0 & 1 \\ \dfrac{q c_{l\alpha} b^2\left(1 + 2e\right) - k_\theta}{I_{\rm EA}} & \dfrac{q c_{l\alpha} b^3\left(\frac{1}{2} - 2e^2\right)}{I_{\rm EA} V} \end{bmatrix}.$$

Notice that entry $A_{22}$ is positive when $|e| < 1/2$: a positive pitch rate $\dot{\theta}$ directly accelerates $\ddot{\theta}$ in the same direction, which is the algebraic signature of negative aerodynamic damping.

Let's define the system parameters and the function that builds $\boldsymbol{A}$. As we did in Notebook 3, we group the model parameters into a `dataclass` to keep them organised, and we will reuse the same container to analyse the system for three different elastic-axis positions.

### Model parameters

Exactly as in Notebook 3, we collect the model parameters in a `dataclass` (a lightweight container that auto-generates the constructor and a readable representation), so we will not dwell on it again here. The baseline configuration for this notebook is:

- Chord $c = 2.9$ m
- Mass moment of inertia $I_{\rm EA} = 223.0$ kgm<sup>2</sup>
- Torsional stiffness $k_\theta = 3.0\times10^4$ Nm/rad
- Nondimensional elastic axis position $e = -0.25$
- Lift curve slope $c_{l\alpha} = 2\pi$ 1/rad (thin airfoil theory)

The air density $\rho$ is kept as a separate argument (it is a property of the flow, set by altitude, not of the section), defaulting to the sea-level value $\rho = 1.225$ kg/m<sup>3</sup>. We also add a small helper, `divergence_speed`, that returns the divergence speed of a given configuration (and `np.inf` when the section cannot diverge). We will use it shortly to size the airspeed ranges of our plots.

In [ ]:
from dataclasses import dataclass


@dataclass
class PitchParams:
    """Physical parameters for the 1-DOF pitch aeroelastic model."""

    I_ea: float = 223.0  # mass moment of inertia [kg*m^2] (Boeing 737-sized section)
    k_theta: float = 3.0e4  # torsional stiffness [N*m/rad] (tuned to keep the example speeds subsonic)
    c: float = 2.9  # chord length [m] (Boeing 737-sized section)
    e: float = -0.25  # elastic axis position from mid-chord [semichords], positive toward the trailing edge
    cl_alpha: float = 2 * np.pi  # lift curve slope [1/rad]


def build_system_matrix(params, velocity, rho=1.225):
    """
    Constructs the system matrix A for the pitch-only typical section model.

    Parameters
    ----------
    params : PitchParams
        Physical parameters of the typical section.
    velocity : float
        Freestream speed [m/s]
    rho : float, optional
        Air density [kg/m^3], by default 1.225 (sea level standard)

    Returns
    -------
    system_matrix : ndarray, shape (2, 2)
        State-space matrix A such that d/dt [theta, theta_dot] = A @ [theta, theta_dot]
    """
    # Calculate semichord
    b = params.c / 2

    # Calculate dynamic pressure from flow conditions
    q = 0.5 * rho * velocity**2

    # Row 1: [0, 1]   (definition: d(theta)/dt = theta_dot)
    # Row 2: [A21, A22]
    #   A21 = (aero stiffness - structural stiffness) / I_ea
    #   A22 = (aero damping numerator) / (I_ea * V)  [positive = negative physical damping]
    A21 = (
        q * params.cl_alpha * b**2 * (1 + 2 * params.e) - params.k_theta
    ) / params.I_ea
    A22 = (
        q
        * params.cl_alpha
        * b**3
        * (1 / 2 - 2 * params.e**2)
        / (params.I_ea * velocity)
    )

    # Assemble the system matrix
    system_matrix = np.array([[0.0, 1.0], [A21, A22]])

    # Return the system matrix
    return system_matrix


def divergence_speed(params, rho=1.225):
    """
    Divergence speed of the 1-DOF pitch section (from the aero-reduced-stiffness condition of Section 2).

    Returns np.inf when the elastic axis is ahead of the aerodynamic center (1 + 2e <= 0): in that
    configuration the aerodynamic moment stiffens the section instead of softening it, so static
    divergence can never occur.

    Parameters
    ----------
    params : PitchParams
        Physical parameters of the typical section.
    rho : float, optional
        Air density [kg/m^3], by default 1.225 (sea level standard).

    Returns
    -------
    float
        Divergence speed [m/s], or np.inf if the section cannot diverge.
    """
    b = params.c / 2
    aero_stiffness_factor = 1 + 2 * params.e
    if aero_stiffness_factor <= 0:
        return np.inf
    return np.sqrt(2 * params.k_theta / (rho * params.cl_alpha * b**2 * aero_stiffness_factor))

## 4. Root Locus Analysis

---

The equation of motion of Section 2 pointed to **three qualitatively different regimes**, depending on where the elastic axis sits. We now study one representative case of each:

- $e = -0.6\rightarrow$ **ahead of the quarter chord** ($e<-1/2$): positive aerodynamic damping and no divergence, so we expect it to be stable at every speed;
- $e = -0.25\rightarrow$ **between the quarter and three-quarter chord** ($|e|<1/2$, the baseline default): negative aerodynamic damping, so we expect it to be unstable;
- $e = 0.6\rightarrow$ **behind the three-quarter chord** ($e>1/2$): positive aerodynamic damping but, being aft of the aerodynamic center, prone to static divergence.

Rather than re-simulating the time response for every speed, we study the dynamics far more efficiently through the **eigenvalues** of the system matrix $\boldsymbol{A}$. Let's recall their meaning:

* $\mathbb{Re}(\lambda)$: the rate of decay (if negative) or growth (if positive) of the motion.
* $\mathbb{Im}(\lambda)$: the oscillation frequency.

| $\mathbb{Re}(\lambda)$ | $\mathbb{Im}(\lambda)$ | Physical meaning |
|---|---|---|
| $< 0$ | $\neq 0$ | Stable, damped oscillation |
| $> 0$ | $\neq 0$ | Unstable, growing oscillation |
| $< 0$ | $= 0$ | Stable, non-oscillatory (overdamped) |
| $> 0$ | $= 0$ | **Unstable, non-oscillatory (static divergence)** |

Watch that last row carefully when reading the root loci: **when a pair of complex-conjugate eigenvalues meets on the real axis and one branch moves into the right half-plane ($\mathbb{Re}(\lambda)>0$, $\mathbb{Im}(\lambda)=0$), the system is in static divergence**. In other words, the dynamic, oscillatory picture has given way to the monotonic divergence of Notebook 1.

To choose a sensible airspeed range for each root locus, we first compute the **divergence speed** of each case:

$$V_D = \sqrt{\frac{q_D}{\frac{1}{2}\rho}} = \sqrt{\frac{2 k_\theta}{\rho\, c_{l\alpha}\, b^2\left(1 + 2e\right)}},$$

which our `divergence_speed` helper evaluates (returning $\infty$ for the $e=-0.6$ case, where $1+2e<0$ and divergence cannot occur). We store everything for each case in a small dictionary and collect the three dictionaries in a list, so we can loop over the cases cleanly throughout the rest of the notebook.

In [ ]:
# Air density at sea level
rho = 1.225  # kg/m^3

# We describe each of the three elastic-axis cases with a small dictionary, and collect the three
# dictionaries in a list. This keeps every quantity associated with a case (its parameters, its label,
# its divergence speed, the sweep range, and later its eigenvalue arrays) together in one place.
# The middle case uses the default PitchParams() (e = -0.25), our baseline 737-derived configuration.
cases = [
    {"params": PitchParams(e=-0.6), "label": "EA ahead of the quarter chord"},
    {"params": PitchParams(), "label": "EA between the quarter and three-quarter chord (baseline)"},
    {"params": PitchParams(e=0.6), "label": "EA behind the three-quarter chord"},
]

# For each case, compute the divergence speed and a sensible top speed for the root-locus sweep:
#   - V_D is finite only when the EA is aft of the AC (1 + 2e > 0); otherwise divergence_speed returns inf.
#   - V_max goes just past divergence where it exists, and falls back to a fixed high speed otherwise.
for case in cases:
    case["V_D"] = divergence_speed(case["params"], rho)
    case["V_max"] = 1.1 * case["V_D"] if np.isfinite(case["V_D"]) else 200.0
    vd_text = f"{case['V_D']:.1f} m/s" if np.isfinite(case["V_D"]) else "infinite (no divergence)"
    print(f"e = {case['params'].e:+.2f}  |  {case['label']:<58s}  |  V_D = {vd_text}")

We are now ready to compute the root loci. We loop over the three cases, and for each of them we compute the eigenvalues of $\boldsymbol{A}$ across a range of flight speeds from $0$ to $1.1 V_D$. We then plot the root locus in the complex plane, with $\mathbb{Re}(\lambda)$ on the x-axis and $\mathbb{Im}(\lambda)$ on the y-axis.

In [ ]:
# Number of flight speeds to evaluate along each root locus
num_speeds = 200

# Iterate over the three cases, computing and plotting one root locus each
for case in cases:

    # Sweep the flight speed from near zero up to this case's ceiling V_max
    velocities = np.linspace(1, case["V_max"], num_speeds)

    # Pre-allocate flat arrays for the real and imaginary parts (2 eigenvalues per speed)
    real_parts = np.zeros(num_speeds * 2)
    imag_parts = np.zeros(num_speeds * 2)

    # Compute the eigenvalues of the system matrix at each speed
    for j, V in enumerate(velocities):
        A = build_system_matrix(case["params"], V)
        eigenvalues = np.linalg.eigvals(A)
        real_parts[j * 2 : (j + 1) * 2] = eigenvalues.real
        imag_parts[j * 2 : (j + 1) * 2] = eigenvalues.imag

    # Store the sweep results inside the case dictionary so the V-f/V-g section can reuse them
    case["velocities"] = velocities
    case["real_parts"] = real_parts
    case["imag_parts"] = imag_parts

    # Scatter plot: colour encodes flight speed
    plt.figure()
    sc = plt.scatter(
        real_parts,
        imag_parts,
        c=np.repeat(velocities, 2),  # repeat each speed twice to match the eigenvalue pairs
    )
    plt.xlabel("$\\mathbb{Re}(\\lambda)$ — exponential growth/decay rate")
    plt.ylabel("$\\mathbb{Im}(\\lambda)$ — oscillation frequency (rad/s)")
    plt.colorbar(sc, label="$V$, m/s")
    plt.grid(True)
    plt.title(f"Root locus for $e = {case['params'].e:.2f}$")
    plt.show()

Let's read the three root locus plots together. First of all, all root loci start near the imaginary axis at very low speed, which is consistent with the fact that aerodynamics is the only source of damping in our system, and with no airflow we have no damping. This is the same behavior that we observed for the heave model in Notebook 2.

For the case with $e=-0.6$, the eigenvalues are always in the **left** half of the complex plane, confirming that the system is stable for all flight speeds. As the flight speed increases, the magnitude of both the real and imaginary parts grows, so the system becomes more damped and oscillates faster. There is no divergence, consistent with $V_D=\infty$ for this configuration.

The root locus for the case with $e=-0.25$ shows a very different behavior. At least one eigenvalue is **always** in the **right** half of the complex plane, confirming that the system is unstable at every speed (negative aerodynamic damping). The eigenvalues start as a complex-conjugate pair whose imaginary part shrinks quickly as speed increases while the real part grows more slowly. Then, **at $V_D$ the two branches meet on the real axis and one of them stays in the right half-plane**: from this speed on, the response is non-oscillatory with a positive real eigenvalue, i.e. **static divergence**. We will confirm this oscillatory-then-monotonic growth in the time histories.

Finally, for the case with $e=0.6$ we see a "complementary" behavior. The eigenvalues start as a complex-conjugate pair in the **left** half of the complex plane (stable, oscillatory). As the speed increases, their imaginary part shrinks while the real part becomes a little more negative, until again **the pair collapses onto the real axis at $V_D$ and one branch crosses into the right half-plane**, the same static-divergence signature, except that here it interrupts an otherwise *stable* oscillatory response. Note how much lower this divergence speed is than for the $e=-0.25$ case, because the elastic axis is much further aft of the aerodynamic center.

## 5. V-f and V-g plots

---

Let's now produce the V-f and V-g plots for the pitch model. As we saw in Notebook 2, they essentially compress the root locus into two scalar curves, which are those used routinely by aeroelasticians. We follow exactly the same procedure as in Notebook 2, using the following formulas to compute the oscillation frequency and the damping ratio at each velocity:

- **Frequency (V-f):** $\omega = \mathbb{Im}(\lambda)$
- **Damping ratio (V-g):** $\zeta = \frac{-\mathbb{Re}(\lambda)}{\sqrt{\mathbb{Re}(\lambda)^2 + \mathbb{Im}(\lambda)^2}}$

Remember that the system is **stable** if $\zeta > 0$, and **unstable** if $\zeta < 0$.

We produce a pair of plots for each of the three cases, with the **V-f diagram on top** and the **V-g diagram below**, reusing the eigenvalues we already computed for the root locus.

In [ ]:
# For each case, turn the eigenvalues stored during the root-locus sweep into frequency and
# damping-ratio curves, then plot them with the V-f diagram on top and the V-g diagram below.
for case in cases:

    velocities = case["velocities"]

    # There are 2 eigenvalues per speed (a complex-conjugate pair for an oscillatory response).
    # We keep one eigenvalue of each pair by taking every other entry ([::2]).
    real = case["real_parts"][::2]
    imag = case["imag_parts"][::2]

    # Frequency (V-f) is the imaginary part; damping ratio (V-g) is -Re / |lambda|
    frequencies = np.abs(imag)
    damping_ratios = -real / np.sqrt(real**2 + imag**2)

    # Two stacked subplots sharing the velocity axis
    fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)

    # Top: V-f diagram (frequency vs velocity)
    ax1.plot(velocities, frequencies)
    ax1.set_ylabel("$\\omega$, rad/s")
    ax1.grid(True)

    # Bottom: V-g diagram (damping ratio vs velocity)
    ax2.plot(velocities, damping_ratios)
    ax2.set_ylabel("$\\zeta$")
    ax2.set_xlabel("$V$, m/s")
    ax2.grid(True)

    # Set title, adjust layout, and show the plots
    plt.suptitle(f"V-f and V-g plots for $e = {case['params'].e:.2f}$")
    plt.tight_layout()
    plt.show()

These three plot pairs summarise the stability picture of the pitch model:

- **$e = -0.6$:** $\omega$ increases (stiffer) and $\zeta > 0$ and increasing (more stable). Unconditionally stable: the $\zeta = 0$ boundary is never crossed.

- **$e = -0.25$:** $\zeta$ starts at a slightly negative value and remains below zero for all speeds (always dynamically unstable). The frequency decreases with speed and collapses to zero at $V_D$, where $\zeta$ jumps discontinuously. This jump is a plotting artefact of the transition from complex to real eigenvalues at divergence, can you explain why?

- **$e = +0.6$:** $\zeta$ starts positive (dynamically stable) and remains positive up to $V_D$, at which point the frequency collapses to zero and an analogous jump of $\zeta$ occurs.

Notice how, in both diverging cases, the **frequency dropping to zero** and the **damping curve breaking** mark exactly the same event: the eigenvalues becoming real with one of them positive, which corresponds to static divergence.

## 6. Time-Domain Simulations

---

The root locus and the V-f/V-g diagrams told us *whether and when* each configuration is stable. To actually **see** the corresponding motion, we now integrate the equation of motion in time for the same three cases.

Analogously to Notebook 2, we will use the `odeint` function from the `integrate` module of the `scipy` library. To use `odeint`, we first need to define a function that computes the time derivative of the state vector, $\dot{\boldsymbol{x}}$, given the current state $\boldsymbol{x}$ and the system matrix $\boldsymbol{A}$.

In [ ]:
from scipy.integrate import odeint


def state_space_model(x, time_array, system_matrix):
    """
    Returns the time derivative of the state vector for the pitch-only model.

    Parameters
    ----------
    x : array [theta, theta_dot]
    time_array : float  (required by odeint but unused for LTI systems)
    system_matrix : ndarray  (2x2 system matrix A)

    Returns
    -------
    dxdt : array  (time derivative of state vector)
    """
    dxdt = system_matrix @ x
    return dxdt

Now we are going to simulate the response to a small initial disturbance: the airfoil starts at $\theta\left(t=0\right) = 2$ degrees with $\dot{\theta}\left(t=0\right) = 0$ rad/s, and we watch what happens. We start by considering an elastic axis position of $e = -0.6$, so just **forward of the quarter chord**, where we expect the system to always be dynamically stable. Analogously to Notebook 2, we simulate the response at three different flight speeds: 40 m/s, 80 m/s, and 160 m/s. We will plot the time response of $\theta(t)$ for each flight speed. 

In [ ]:
# Time vector and initial disturbance: theta = 2 degrees, theta_dot = 0 rad/s
t = np.linspace(0, 15, 1000)  # simulate for 15 seconds with 1000 time points
initial_condition = [np.radians(2), 0.0]

# First case: EA ahead of the quarter chord (e = -0.6), expected stable at all speeds
case = cases[0]
flight_speeds = [40, 80, 160]  # m/s

# Create figure
plt.figure()

# Iterate over speeds
for V in flight_speeds:

    # Build the system matrix for this speed and solve the ODE
    A = build_system_matrix(case["params"], V)
    solution_array = odeint(state_space_model, initial_condition, t, args=(A,))

    # Extract 'theta' (the first column of the solution) and plot it in degrees
    theta_response = np.degrees(solution_array[:, 0])
    plt.plot(t, theta_response, label=f"$V = {V}$ m/s")

# Set axes labels, legend, and grid
plt.xlabel("$t$, s")
plt.ylabel("$\\theta$, deg")
plt.legend()
plt.grid(True)
plt.title(f"Time response of pitch model for $e = {case['params'].e:.2f}$")
plt.show()

As expected for $e = -0.6$, the **system is stable for all flight speeds**, and the **oscillations are more damped at higher flight speeds**, where the aerodynamic damping is stronger.

A notable difference with the heave model is that the **oscillation frequency changes significantly with flight speed**. Can you explain why this happens? Let's go back at the mass-spring-damper analogy to find the answer.

In Notebook 2, we reviewed the general solution of the mass-spring-damper system, and we found that the general solution is characterized by the eigenvalues of the characteristic equation. In the case of underdamped response, these eigenvalues can be written as:

$$
\lambda_{1,2} = -\zeta \omega_0 \pm j \omega_0 \sqrt{1-\zeta^2},
$$

where $\omega_0=\sqrt{k/m}$ is the natural frequency of the undamped system, $\zeta=c/2\left(mk\right)^{1/2}$ is the damping ratio, and $k$, $m$, and $c$ are the stiffness, mass, and damping coefficients, respectively. The oscillation frequency of the damped system is given by the imaginary part of the eigenvalues:

$$\omega = \omega_0 \sqrt{1-\zeta^2}.$$

In the heave model, the stiffness and the mass terms were constant, so the only contribution to the change in oscillation frequency with flight speed came from the damping ratio $\zeta$, which typically has a small value for underdamped motion. In the pitch model, instead, the stiffness term includes both an elastic and an aerodynamic component, and the latter changes significantly with flight speed, which leads to a much larger change in the oscillation frequency of the system.

In particular, with $e<-0.5$, the aerodynamic stiffness term $- q c_{l\alpha} b^2\left(1 + 2e\right)$ is positive, and it increases with flight speed. This leads to an increase in the overall stiffness of the system with flight speed, which in turns leads to an increase in the oscillation frequency, as indeed we observe in the time response plots, and as the V-f diagram for $e=-0.6$ already showed.

This is something very important to keep in mind: **the oscillation frequencies of aeroelastic systems can change significantly with flight speed**. This is far from a cosmetic detail, as it lies at the very heart of the flutter you saw in Notebook 3, where the heave and pitch frequencies approach one another as the speed increases. Note how the change in frequency due to pitch rate effects is much larger than that obtained considering heave velocity only, suggesting that it is important to include these effects in the model if we want to capture the flutter correctly.

Let's now turn to the **baseline case**, with the elastic axis between the quarter and three-quarter chord ($e = -0.25$), which the root locus told us is **unstable at every speed**. We reuse the divergence speed $V_D$ we computed in Section 4 to pick three telling flight speeds: $0.5\,V_D$, $0.99\,V_D$, and $1.01\,V_D$.

🤔 **Predict before you run.** With $e=-0.25$ the elastic axis sits *inside* the $|e|<1/2$ band, so the aerodynamic damping is negative. Below the divergence speed, do you expect the pitch oscillations to decay, hold steady, or grow? And what qualitatively different behaviour do you expect once the speed exceeds $V_D$? Make a prediction, then run the cells below and check it.

In [ ]:
# Second case: EA between the quarter and three-quarter chord (baseline, e = -0.25), expected unstable
case = cases[1]

# Flight speeds as fractions of this case's divergence speed (computed in Section 4)
V_D_multipliers = [0.5, 0.99, 1.01]
flight_speeds = [multiplier * case["V_D"] for multiplier in V_D_multipliers]

# Create figure
plt.figure()

# Iterate over speeds
for multiplier, V in zip(V_D_multipliers, flight_speeds):

    # Build the system matrix for this speed and solve the ODE
    A = build_system_matrix(case["params"], V)
    solution_array = odeint(state_space_model, initial_condition, t, args=(A,))

    # Extract 'theta' (the first column of the solution) and plot it in degrees
    theta_response = np.degrees(solution_array[:, 0])
    plt.plot(
        t, theta_response, label=f"$V = {multiplier:.2f} \\cdot V_D = {V:.1f}$ m/s"
    )

# Set axes labels, legend, and grid
plt.xlabel("$t$, s")
plt.ylabel("$\\theta$, deg")
plt.legend()
plt.grid(True)
plt.title(f"Time response of pitch model for $e = {case['params'].e:.2f}$")
plt.show()

What's happening here? We can only see the curve at $1.01\cdot V_D$, with values of $\theta$ that grow exponentially to astronomically large values, many orders of magnitude beyond any physically meaningful angle! This is a clear sign of static instability: the system is diverging in a non-oscillatory way, as we would expect for a flight speed greater or equal to the divergence speed. Below $V_D$, instead, the oscillations grow (the system is dynamically unstable), exactly the oscillatory-then-monotonic picture the root locus predicted.

To visualize the other two curves at $0.5\cdot V_D$ and $0.99\cdot V_D$, try to add `plt.ylim(-40, 40)` before the `plt.show()` command. What do you see? You should see the curves at $0.5\cdot V_D$ and $0.99\cdot V_D$ diverging in an oscillatory way, and with a noticeably different oscillation frequency. Which curve diverges faster? And which one has a higher oscillation frequency? Can you explain why?

Finally, let's examine the case with the **elastic axis just behind the three-quarter chord** $\left(e=0.6\right)$, which the root locus told us is dynamically stable yet still subject to static divergence. We reuse its (much lower) divergence speed from Section 4. Let's simulate the response at the same three fractions of the divergence speed.

In [ ]:
# Third case: EA behind the three-quarter chord (e = 0.6), dynamically stable but able to diverge
case = cases[2]

# Same fractions of this case's (much lower) divergence speed
flight_speeds = [multiplier * case["V_D"] for multiplier in V_D_multipliers]

# Create figure
plt.figure()

# Iterate over speeds
for multiplier, V in zip(V_D_multipliers, flight_speeds):

    # Build the system matrix for this speed and solve the ODE
    A = build_system_matrix(case["params"], V)
    solution_array = odeint(state_space_model, initial_condition, t, args=(A,))

    # Extract 'theta' (the first column of the solution) and plot it in degrees
    theta_response = np.degrees(solution_array[:, 0])
    plt.plot(
        t, theta_response, label=f"$V = {multiplier:.2f} \\cdot V_D = {V:.1f}$ m/s"
    )

# Set axes labels, legend, and grid
plt.xlabel("$t$, s")
plt.ylabel("$\\theta$, deg")
plt.legend()
plt.grid(True)
plt.title(f"Time response of pitch model for $e = {case['params'].e:.2f}$")
plt.show()

Once again we can only see the curve at $1.01\cdot V_D$ showing an exponential growth of $\theta$, indicating the expected occurrence of divergence. To visualize the other two curves at $0.5\cdot V_D$ and $0.99\cdot V_D$, try to add `plt.ylim(-5, 5)` before the `plt.show()` command. You should see the response at $0.5\cdot V_D$ and $0.99\cdot V_D$ decaying in an oscillatory way, meaning that the system is dynamically stable. Once again you will see an observable difference both in the rate of decay and in the oscillation frequency as we increase the flight speed. Which curve decays faster? And which one has a higher oscillation frequency? Can you explain why?

Something interesting to notice for the case with $e>0.5$ is that, while the system is dynamically stable, it can still experience static instability at a certain flight speed. In some way, the transition from the case with $-0.5<e<0.5$ to the case with $e>0.5$ hints at a typical feature of aircraft wings, which is that while moving the elastic axis further aft of the aerodynamic center can benefit dynamic stability, it will generally be detrimental to static stability. This means that the position of the elastic axis cannot be chosen in such a way to benefit both dynamic and static stability at the same time, and the aircraft designer must find a compromise between the two. However, we should also keep in mind that in reality the position of the elastic axis cannot be chosen freely for aircraft wings, as it is determined by their structural design, which is driven by many different factors.

> **Key insight:** the pitch-only model has *two* distinct instability mechanisms: an intrinsic **dynamic** instability (growing oscillations from negative aerodynamic damping) and a **static** instability (divergence). While in theory it is possible to suppress both by placing the elastic axis ahead of the quarter chord, as discussed in Notebook 1, this is not a feasible solution for real aircraft wings.

## 7. Exercise — The Role of the Elastic Axis Position

---

In the sections above we explored three specific values of $e$. Now your task is to look at the picture more systematically by sweeping $e$ continuously from $-1$ to $+1$ at different fixed flight speeds and tracking the eigenvalues. You must fill the code cell below, produce a root locus coloured by $e$, and then interpret the results.

Since `build_system_matrix` takes a `PitchParams` object, you can create a new parameter set for each value of $e$ inside your loop:

```python
V = 40.0  # m/s
for e_val in np.linspace(-1, 1, 50):
    params = PitchParams(e=e_val)
    A = build_system_matrix(params, V)
    eigenvalues = np.linalg.eigvals(A)
    # ... store and plot
```

Start with a flight speed of 40 m/s. Does the root locus cross the imaginary axis at some value of $e$? If so, does it cross it for the values of $e$ that you would expect? Can you identify and characterize different regions of stability (stable vs unstable, oscillatory vs non-oscillatory) as a function of $e$? Can you explain the physical mechanisms behind these different regions?

Successively, increase the flight speed to 60 m/s. Do you see any difference in the crossings of the imaginary axis and in the stability regions compared to the case with 40 m/s? Can you explain these differences based on the physics of the system?

In [ ]:
# Select a flight speed and calculate dynamic pressure
# ...

# Define range of elastic axis locations to consider for the root locus plot
# ...

# Pre-allocate arrays for real and imaginary parts (2 eigenvalues per speed)
# ...

# Iterate over elastic axis positions
# ...
    # Build system matrix and compute eigenvalues
    # ...

    # Store real and imaginary parts for both eigenvalues at this speed
    # ...

# Produce scatter plot with colour encoding elastic axis position
# ...

## 8. Recap & Conclusion: The Pitch-Only Model in Perspective

---

Congratulations on completing Notebook 4! Let us summarise what we have discovered and, just as importantly, reflect on what our model **cannot** capture.

### Key Takeaways

1. **Spatially varying angle of attack.** Unlike heave, pitch rotation creates a different relative wind direction at every chord point. The corresponding effective angle of attack varies linearly along the chord, and we need to determine a single point where to evaluate it.

2. **The three-quarter chord collocation.** Thin-airfoil theory provides the answer to which point to use: place the bound vortex at the quarter chord and enforce flow tangency at the three-quarter chord. This lumped vortex element exactly reproduces the correct thin-airfoil lift. The effective angle of attack for the pitch model is therefore:
$$\alpha = \theta + \frac{b}{V}\left(\frac{1}{2} - e\right)\dot{\theta}.$$

3. **Sign of aerodynamic damping depends on $e$ and does not change with $V$.** The pitch-rate term modifies the lift, which acts at the quarter chord. Whether the resulting moment reinforces or opposes the rotation depends on the position of the elastic axis with respect to the aerodynamic center and the three-quarter chord:
   - $|e| < 1/2$: **negative damping** — the system is always unstable.
   - $|e| > 1/2$: **positive damping** — the system is always stable.

The damping *never crosses zero* with speed, so we do not have flutter in this model.

4. **Two instability mechanisms.** The pitch model can lose stability through a **dynamic** instability (growing oscillations from negative aerodynamic damping) or a **static** instability (monotonic divergence). These are distinct phenomena and their occurrence depends on $e$ and flight speed.

5. **Aeroelastic frequencies shift with speed.** The effective stiffness of the system includes an aerodynamic contribution that changes with the dynamic pressure. Depending on the value of $e$, speed can either increase or decrease the oscillation frequency, something much less evident in a pure translational (heave) model.


### What Quasi-Steady Aerodynamics Discards — and Why It Matters

The missing ingredient is the **memory of the flow**. When a real airfoil oscillates, it continuously sheds vorticity into its wake. That wake does not disappear: it keeps inducing a velocity field back onto the airfoil, altering its effective angle of attack in a way that depends on the *history* of the motion, not just the instantaneous state. This introduces a **phase lag** between the motion and the resulting aerodynamic force, and the size of that lag depends on the **reduced frequency**, and hence on airspeed. Quasi-steady aerodynamics, by construction, sees only the instantaneous state and is blind to this mechanism.

How wrong can quasi-steady be for the pitch-only section? [In a landmark 1949 paper](https://arc.aiaa.org/doi/full/10.2514/8.11885), Smilg analysed exactly this problem with Theodorsen's full unsteady theory and reached a conclusion our model cannot even represent: a pitch-only airfoil **can** flutter, and it does so in a configuration that quasi-steady declares perfectly safe, namely with the **elastic axis ahead of the quarter chord** ($e<-1/2$), where our model predicts positive damping *and* no divergence. Quasi-steady's reassuring verdict for that configuration is simply **wrong**.

To be honest about the size of the effect: Smilg's single-DOF pitch flutter requires fairly extreme conditions: the rotation axis ahead of the quarter chord (but not too far ahead of the leading edge) **and** a very large non-dimensional inertia, $I_\alpha/(\pi\rho b^4)\gtrsim 550$. The representative section we used here has $I_\alpha/(\pi\rho b^4)$ of order ten, so it would *not* flutter in pure pitch even with unsteady aerodynamics. The lesson is therefore qualitative but fundamental: **quasi-steady aerodynamics can give not merely inaccurate but qualitatively wrong stability predictions**, and we have no way of knowing when without a better aerodynamic theory.


### Two Routes to Flutter

It is worth stepping back to see how flutter can appear based on what we have seen so far in this course:

| Route | Aerodynamics | Degrees of freedom | Source of net positive work |
|---|---|---|---|
| Binary flutter (Notebook 3) | Quasi-steady | 2 — heave + pitch coupled | Aeroelastic coupling shifts the phase between pitch and heave |
| Smilg (1949) | Full unsteady (Theodorsen) | 1 — pitch only | Wake-induced lag shifts the phase of the aerodynamic moment |

You have **already travelled the first route** in Notebook 3: with quasi-steady aerodynamics, *coupling* two degrees of freedom is enough to make the maximum lift align with the heave velocity over a cycle, feed net positive work into the structure, and trigger flutter above a critical speed. The second route needs an aerodynamic phase lag that only unsteady theory can provide, which is exactly where we go next.

One last, sobering point. The wake memory that quasi-steady ignores does not only *create* the single-DOF pitch flutter Smilg found, it also shifts the **flutter speed of the 2-DOF model you already built**. The flutter speed you computed in Notebook 3 is only an approximation. Unsteady aerodynamics is needed both for qualitative correctness **and** for quantitative accuracy.


### Coming Up Next: Unsteady Aerodynamics

In **Notebook 5** we lift the quasi-steady restriction and introduce **Theodorsen's unsteady aerodynamic theory**. We will:
1. Give the flow its memory back through the lift-deficiency function $C(k)$ and the reduced frequency $k$.
2. See how the aerodynamic forces acquire a frequency-dependent phase lag.
3. Revisit the pitch-only section and recover the genuine, speed-dependent stability behaviour that Smilg predicted.
4. Re-examine the 2-DOF flutter speed of Notebook 3 and quantify how much quasi-steady got wrong.

The flow is about to remember its past. 🚀

---
**End of Notebook 4** ✓  
*Save your work and proceed to Notebook 5: Unsteady Aerodynamics and Theodorsen's Theory.*